# LexAI — Notebook 01: Legal Database Setup

This notebook walks through building the Indian legal knowledge base:
- **IPC Sections** (Indian Penal Code)
- **CrPC Sections** (Code of Criminal Procedure)
- **Case Precedents** (Supreme Court landmark cases)
- **Recent Judgments** (Supreme Court & High Courts)

Run `scripts/build_legal_database.py` first, or execute the cells below to explore the data inline.

In [ ]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

DATA_DIR = pathlib.Path('../data/legal_database')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Data directory:', DATA_DIR.resolve())

## 1. Build the Legal Database

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-X', 'utf8', '../scripts/build_legal_database.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 2. Explore IPC Sections

In [ ]:
with open(DATA_DIR / 'ipc_sections.json') as f:
    ipc = json.load(f)

print(f'Total IPC sections: {len(ipc["sections"])}')
print('\nSample sections:')
for s in ipc['sections'][:5]:
    print(f"  IPC {s['section']} — {s['title']} (Severity: {s.get('severity', 'N/A')})")

In [ ]:
import pandas as pd

df_ipc = pd.DataFrame(ipc['sections'])
print(df_ipc[['section', 'title', 'punishment', 'bailable']].head(10).to_string(index=False))

## 3. Explore Case Precedents

In [ ]:
with open(DATA_DIR / 'case_precedents.json') as f:
    precedents = json.load(f)

print(f'Total precedents: {len(precedents["precedents"])}')
print('\nTop 5 landmark cases:')
for p in precedents['precedents'][:5]:
    print(f"  [{p['year']}] {p['case_name']}")
    print(f"    Principle: {p['legal_principle'][:80]}...")
    print()

## 4. Severity Distribution

In [ ]:
import matplotlib.pyplot as plt

severity_counts = df_ipc['severity'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
severity_counts.plot(kind='bar', ax=ax, color='#c9a84c', edgecolor='black')
ax.set_title('IPC Sections by Severity Level', fontsize=14, fontweight='bold')
ax.set_xlabel('Severity (1=Minor, 5=Capital)')
ax.set_ylabel('Count')
ax.set_facecolor('#f8f5ee')
plt.tight_layout()
plt.show()
print('Severity distribution:', severity_counts.to_dict())

## 5. Bailable vs Non-Bailable Offences

In [ ]:
bail_counts = df_ipc['bailable'].value_counts()
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    bail_counts.values,
    labels=['Non-Bailable' if not k else 'Bailable' for k in bail_counts.index],
    colors=['#c0392b', '#27ae60'],
    autopct='%1.0f%%',
    startangle=90
)
ax.set_title('Bailable vs Non-Bailable Offences in Dataset')
plt.show()

## Summary

The legal database contains:
- **IPC sections** with severity ratings, punishment details, bailable status
- **CrPC sections** for procedural law
- **Landmark precedents** from Supreme Court (1954–2023)
- **Recent judgments** for contemporary legal context

This database feeds into the RAG pipeline in Notebook 03.